# TEKO — Search & Docking

This notebooks serves to organize and to explain the training scripts

### Step 1 - Select the path and run settings

Set your repo root and the main entry points (train/eval/export).

In [ ]:

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

import argparse
import sys
import math
import socket
import time
import random
import csv
from collections import deque
from functools import partial
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import optuna
except ImportError:
    print("ERROR: optuna not installed", flush=True)
    sys.exit(1)

try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
except ImportError:
    HAS_TENSORBOARD = False

from isaaclab.app import AppLauncher
print = partial(print, flush=True)

### Step 2 — Optuna configuration

PPO is a policy-gradient method that updates the policy using a surrogate objective to reduce the risk of large performance drops.

Because PPO performance is sensitive to hyperparameter choices, Optuna is used to automatically search over a set of variable hyperparameters and find combinations that best match the defined objective. Optuna is an automatic hyperparameter optimization framework commonly used in machine learning.

---

#### fixed hyperparameters (constant across trials)

- gamma = 0.99  
- value_coef = 0.5  
- max_grad_norm = 0.5  
- clip_ratio = 0.2  
- num_envs = 120  
- rollout_len = 128  
- advance_threshold = 0.85  
- min_steps_before_advance = 200_000  
- max_stage = 49  
- log_interval = 50_000  
- save_interval = 2_000_000  

---

#### variable hyperparameters (optimized by optuna)

- learning_rate ∈ [5e−5, 3e−4] (log)  
- entropy_coef  ∈ [0.003, 0.015] (log)  
- gae_lambda    ∈ [0.90, 0.98]  
- epochs        ∈ {3, 4, 5, 6, 7}  
- batch_size    ∈ {1024, 2048}  
- aux_yaw_coef  ∈ [0.15, 0.45]  

* note: the variable hyperparameters will be used further in the script

---

#### references

- optuna: https://optuna.org/  
- ppo hyperparameters overview: https://medium.com/aureliantactics/ppo-hyperparameters-and-ranges-6fc2d29bccbe


In [ ]:
OPTUNA_CONFIG = {
    # Database Location
    "study_name": "teko_vision_final_v10",
    "storage_path": "sqlite:////home/schux00/optuna/teko_vision_final_v10.db",
    "target_total_trials": 1000,
    "max_steps_per_trial": 300_000_000,  
    "max_walltime_s_per_trial": 120 * 3600, 
    
    # Pruning config for 50 stages
    "pruning_enabled": True,
    "pruning_warmup_steps": 2_000_000,
    "pruning_check_interval": 500_000,
    "stagnation_limit_steps": 30_000_000,  # Prune if stuck at same stage for 30M steps
    "min_stage_schedule": {
        10_000_000: 5,
        20_000_000: 10,
        40_000_000: 18,
        60_000_000: 25,
        100_000_000: 32,
        150_000_000: 38,
        200_000_000: 42,
        250_000_000: 45,
        300_000_000: 47,
        350_000_000: 48,
    },
}


ENTROPY_FLOOR = 0.5  # Minimum entropy to prevent policy collapse
FIXED_PARAMS = {
    "gamma": 0.99,
    "value_coef": 0.5,
    "max_grad_norm": 0.5,
    "clip_ratio": 0.2,
    "num_envs": 120,
    "rollout_len": 128,
    "advance_threshold": 0.85,
    "min_steps_before_advance": 200_000,
    "max_stage": 49,  # UPDATED: 50 stages (0-49)
    "log_interval": 50_000,
    "save_interval": 2_000_000,
}

def should_prune(step, max_stage):
    """Check if trial should be pruned based on progress."""
    if step < OPTUNA_CONFIG["pruning_warmup_steps"]:
        return False
    schedule = OPTUNA_CONFIG["min_stage_schedule"]
    for step_threshold in sorted(schedule.keys()):
        if step >= step_threshold:
            if max_stage < schedule[step_threshold]:
                return True
    return False

### Step 3 — neural network architecture (vision + imu + attention + yaw auxiliary head)

This section defines the policy network used for PPO. The architecture is composed of four main blocks:

- 1 spatial attention: learns a pixel-wise attention mask to emphasize relevant regions in the feature maps.

- 2 channel attention: re-weights feature channels based on their global importance (squeeze-and-excitation style).

- 3 vision encoder: a CNN that processes the 4-channel image input (4 × 128 × 128), followed by attention modules and a fully connected projection to a 256-dim embedding. The encoder also includes an auxiliary yaw prediction head.

- 4 policy head (actor-critic): fuses the vision embedding with an IMU embedding (6 → 64) and produces:
  - actor output: mean action for a 2D continuous control signal, squashed with `tanh`.
  - critic output: state-value estimate using an asymmetric critic input (vision+imu features plus a 7D privileged vector when available).
  - yaw prediction: auxiliary estimate of yaw (scaled to radians) used as an additional learning signal.

key implementation details:
- the actor uses a diagonal Gaussian with learnable `log_std`, clamped to keep exploration stable.
- orthogonal initialization is applied to convolutional layers and the main fully connected layer.
- group normalization is used after each convolution to improve training stability with large batches.
- the yaw head outputs `tanh` and is scaled by `π` to represent angles in radians.


In [ ]:

class SpatialAttention(nn.Module):
    """Spatial attention: learns a per-pixel mask to emphasize relevant regions."""
    def __init__(self, in_channels):
        super().__init__()
         # 1x1 conv -> single attention map (H×W), later broadcast over channels
        self.conv = nn.Conv2d(in_channels, 1, kernel_size=1)
    
    def forward(self, x):
         # sigmoid produces weights in [0,1]
        return x * torch.sigmoid(self.conv(x))


class ChannelAttention(nn.Module):
    """Channel attention (SE-style): re-weights channels based on global context."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        # MLP on global average pooled channels -> channel-wise gates
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, h, w = x.shape
        # global average pooling over spatial dims -> (b, c)
        y = x.view(b, c, -1).mean(-1)
        # reshape back to (b, c, 1, 1) so it broadcasts over H,W
        return x * self.fc(y).view(b, c, 1, 1)


class VisionEncoderAttentionYaw(nn.Module):
    """
    Vision encoder:
    - CNN backbone + group norm
    - channel + spatial attention
    - projection to feature_dim
    - auxiliary yaw prediction head (used for auxiliary loss)
    """    
    def __init__(self, in_channels=4, feature_dim=256):
        super().__init__()
        # Input: (B, 4, 128, 128)
        self.conv1 = nn.Conv2d(in_channels, 32, 8, stride=4, padding=2)  # -> ~32x32
        self.conv2 = nn.Conv2d(32, 64, 4, stride=2, padding=1)           # -> ~16x16
        self.conv3 = nn.Conv2d(64, 64, 3, stride=1, padding=1)           # -> ~16x16
        
        self.channel_attn = ChannelAttention(64)
        self.spatial_attn = SpatialAttention(64)
        
        # GroupNorm is stable for large batches and works well vs BatchNorm in RL
        self.gn1 = nn.GroupNorm(8, 32)
        self.gn2 = nn.GroupNorm(8, 64)
        self.gn3 = nn.GroupNorm(8, 64)
        
        self._init_weights()

        # Infer flattened conv output size automatically (keeps code robust)
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 128, 128)
            flat_size = self._forward_conv(dummy).shape[1]
        
        # Final vision embedding
        self.fc = nn.Linear(flat_size, feature_dim)
        nn.init.orthogonal_(self.fc.weight, gain=1.0)
        nn.init.zeros_(self.fc.bias)
        
        # Auxiliary yaw head: predicts normalized yaw then scaled to radians
        self.yaw_head = nn.Sequential(
            nn.Linear(feature_dim, 64), nn.ReLU(True),
            nn.Linear(64, 32), nn.ReLU(True),
            nn.Linear(32, 1), nn.Tanh()
        )
        self.feature_dim = feature_dim
    
    def _init_weights(self):
        # Orthogonal init tends to improve PPO stability
        for m in [self.conv1, self.conv2, self.conv3]:
            nn.init.orthogonal_(m.weight, gain=nn.init.calculate_gain('relu'))
            nn.init.zeros_(m.bias)
    
    def _forward_conv(self, x):
        x = F.relu(self.gn1(self.conv1(x)))
        x = F.relu(self.gn2(self.conv2(x)))
        x = F.relu(self.gn3(self.conv3(x)))
        x = self.channel_attn(x)   # channel-wise reweighting
        x = self.spatial_attn(x)   # spatial reweighting
        return x.flatten(1)        # (B, flat_size)
    
    def forward(self, x):
        # Returns vision features (B, feature_dim)
        return F.relu(self.fc(self._forward_conv(x)))
    
    def predict_yaw(self, features):
        # yaw in radians ([-pi, pi])
        return self.yaw_head(features) * math.pi


class VisionIMUAttentionYawPolicy(nn.Module):
    """
    PPO policy:
    - actor: vision+imu -> mean action (Gaussian policy)
    - critic: asymmetric critic, optionally uses privileged info (7D)
    - yaw: auxiliary head from vision encoder
    """
    LOG_STD_MIN, LOG_STD_MAX = -2.0, 0.5  # clamp exploration noise
    
    def __init__(self, vis_dim=256, imu_dim=6, hidden=256, action_dim=2):
        super().__init__()
        self.vision_encoder = VisionEncoderAttentionYaw(in_channels=4, feature_dim=vis_dim)
        
        # IMU encoder: (B, 6) -> (B, 64)
        self.imu_encoder = nn.Sequential(
            nn.Linear(imu_dim, 64), nn.ReLU(True),
            nn.Linear(64, 64), nn.ReLU(True),
        )
        
        fused_dim = vis_dim + 64
        
        # Actor outputs mean action; tanh squashing happens later
        self.actor_head = nn.Sequential(
            nn.Linear(fused_dim, hidden), nn.ReLU(True),
            nn.Linear(hidden, action_dim),
        )
        
        # Learnable log std (diagonal Gaussian), shared across states
        self.log_std = nn.Parameter(torch.full((action_dim,), -0.5))
        
        # Asymmetric critic: adds privileged state info when available
        priv_dim = 7
        self.critic_head = nn.Sequential(
            nn.Linear(fused_dim + priv_dim, hidden), nn.ReLU(True),
            nn.Linear(hidden, hidden // 2), nn.ReLU(True),
            nn.Linear(hidden // 2, 1),
        )
    
    def _std(self):
        # Clamp log_std for numerical stability
        return torch.exp(torch.clamp(self.log_std, self.LOG_STD_MIN, self.LOG_STD_MAX))
    
    def forward_features(self, rgb, imu):
        # rgb: (B, 4, 128, 128), imu: (B, 6)
        vis_feat = self.vision_encoder(rgb)     # (B, vis_dim)
        imu_feat = self.imu_encoder(imu)        # (B, 64)
        fused = torch.cat([vis_feat, imu_feat], dim=-1)  # (B, fused_dim)
        return fused, vis_feat
    
    def act(self, rgb, imu, privileged=None, deterministic=False):
        fused, vis_feat = self.forward_features(rgb, imu)
        
        mean = self.actor_head(fused)
        std = self._std().unsqueeze(0).expand_as(mean)
        dist = torch.distributions.Normal(mean, std)
        
        # Reparameterized sample for better gradients through sampling
        u = dist.mean if deterministic else dist.rsample()
        action = torch.tanh(u)  # bound actions to [-1, 1]
        
        # Tanh-squashed Gaussian log prob (change-of-variables correction)
        log_prob = dist.log_prob(u).sum(-1) - torch.log(1 - action.pow(2) + 1e-6).sum(-1)
        
        # Critic input: fused + privileged (or zeros if not provided)
        if privileged is not None:
            critic_in = torch.cat([fused, privileged], dim=-1)
        else:
            critic_in = torch.cat([fused, torch.zeros(fused.shape[0], 7, device=fused.device)], dim=-1)
        
        value = self.critic_head(critic_in).squeeze(-1)
        yaw_pred = self.vision_encoder.predict_yaw(vis_feat)
        return action, log_prob, value, yaw_pred
    
    def evaluate(self, rgb, imu, actions, privileged=None):
        # Used during PPO updates to recompute log_prob, entropy, and value
        fused, vis_feat = self.forward_features(rgb, imu)
        
        mean = self.actor_head(fused)
        std = self._std().unsqueeze(0).expand_as(mean)
        dist = torch.distributions.Normal(mean, std)
        
        # Inverse tanh to recover pre-squash action u
        u = torch.clamp(actions, -0.999, 0.999)
        u = 0.5 * (torch.log1p(u) - torch.log1p(-u))
        
        log_prob = dist.log_prob(u).sum(-1) - torch.log(1 - actions.pow(2) + 1e-6).sum(-1)
        entropy = dist.entropy().sum(-1)
        
        if privileged is not None:
            critic_in = torch.cat([fused, privileged], dim=-1)
        else:
            critic_in = torch.cat([fused, torch.zeros(fused.shape[0], 7, device=fused.device)], dim=-1)
        
        value = self.critic_head(critic_in).squeeze(-1)
        yaw_pred = self.vision_encoder.predict_yaw(vis_feat)
        return log_prob, value, entropy, yaw_pred

### Step 4 — PPO update and advantage estimation

This section implements the core PPO learning functions used during training:

- `compute_gae(...)` computes generalized advantage estimation (GAE) from rollout rewards, value predictions, and terminal flags. It returns:
  - `advantages`: discounted advantage estimates
  - `returns`: bootstrapped targets for value learning (`advantages + values`)

- `ppo_update_with_yaw(...)` performs the PPO optimization step using the collected rollout:
  - flattens `(T, N, ...)` rollouts into a single batch of size `T×N`
  - normalizes advantages to stabilize updates
  - runs multiple epochs of minibatch SGD (shuffle + minibatches)
  - computes PPO clipped surrogate loss (`p_loss`) and value loss (`v_loss`)
  - adds an auxiliary yaw regression loss (`yaw_loss`) to support representation learning
  - includes an entropy term to encourage exploration, with an additional penalty if entropy drops below `ENTROPY_FLOOR`
  - applies gradient clipping (`max_grad_norm`) for stability

The function returns averaged metrics (policy loss, value loss, entropy, yaw loss, gradient norm) for logging.


In [ ]:
def compute_gae(rewards, values, dones, gamma, lam, last_value):
    """
    Generalized Advantage Estimation (GAE).
    
    rewards: (T, N) tensor
    values : (T, N) tensor (V(s_t))
    dones  : (T, N) tensor (1 if episode ended at t, else 0)
    last_value: (N,) bootstrap value for final state after rollout
    
    Returns:
      advantages: (T, N)
      returns   : (T, N) = advantages + values (target for critic)
    """
    T, N = rewards.shape
    advantages = torch.zeros_like(rewards)
    last_gae = torch.zeros(N, device=rewards.device)

    # Work backwards to efficiently compute discounted sums
    for t in reversed(range(T)):
        # Bootstrap with last_value at final step; otherwise use next timestep value
        next_val = last_value if t == T - 1 else values[t + 1]

        # TD error (delta)
        delta = rewards[t] + gamma * next_val * (1 - dones[t]) - values[t]

        # GAE recursion (stop accumulating across episode boundaries using dones)
        last_gae = delta + gamma * lam * (1 - dones[t]) * last_gae
        advantages[t] = last_gae

    returns = advantages + values
    return advantages, returns


def ppo_update_with_yaw(
    policy, optimizer,
    rgb, imu, actions, old_logp,
    advantages, returns,
    yaw_targets, privileged,
    cfg
):
    """
    PPO update with:
    - clipped surrogate objective
    - value function regression
    - entropy bonus + entropy floor penalty
    - auxiliary yaw regression loss
    """
    device = next(policy.parameters()).device

    # Rollout shapes: (T, N, ...)
    T, N = rgb.shape[:2]
    total = T * N

    # Flatten rollouts into a single batch (total, ...)
    rgb_flat = rgb.view(total, *rgb.shape[2:])
    imu_flat = imu.view(total, -1)
    actions_flat = actions.view(total, -1)
    old_logp_flat = old_logp.view(total)
    ret_flat = returns.view(total)
    yaw_flat = yaw_targets.view(total, 1)

    # Normalize advantages (standard PPO stabilization trick)
    adv_flat = advantages.view(total)
    adv_flat = (adv_flat - adv_flat.mean()) / (adv_flat.std() + 1e-8)

    # Privileged input is optional (asymmetric critic)
    priv_flat = privileged.view(total, -1) if privileged is not None else None

    # Track averaged metrics
    metrics = {"policy_loss": 0, "value_loss": 0, "entropy": 0, "yaw_loss": 0, "grad_norm": 0}
    n_updates = 0

    # Multiple epochs over the same rollout buffer
    for _ in range(cfg["epochs"]):
        # Shuffle indices for minibatch SGD
        idx = torch.randperm(total, device=device)

        for start in range(0, total, cfg["batch_size"]):
            mb = idx[start:start + cfg["batch_size"]]
            priv_mb = priv_flat[mb] if priv_flat is not None else None

            # Recompute log-prob, value, entropy under current policy
            logp, val, ent, yaw_pred = policy.evaluate(
                rgb_flat[mb], imu_flat[mb], actions_flat[mb], priv_mb
            )

            # PPO ratio = pi(a|s) / pi_old(a|s)
            ratio = torch.exp(logp - old_logp_flat[mb])

            # Clipped surrogate objective
            surr1 = ratio * adv_flat[mb]
            surr2 = torch.clamp(ratio, 1 - cfg["clip_ratio"], 1 + cfg["clip_ratio"]) * adv_flat[mb]
            p_loss = -torch.min(surr1, surr2).mean()

            # Critic loss (MSE on returns)
            v_loss = 0.5 * F.mse_loss(val, ret_flat[mb])

            # Auxiliary yaw regression loss (improves representation learning)
            yaw_loss = F.mse_loss(yaw_pred, yaw_flat[mb])

            # Entropy bonus encourages exploration (maximize entropy => minimize -entropy)
            ent_mean = ent.mean()
            entropy_loss = -cfg["entropy_coef"] * ent_mean

            # Extra penalty if entropy collapses below a minimum floor
            if ent_mean.item() < ENTROPY_FLOOR:
                entropy_loss -= 0.1 * (ENTROPY_FLOOR - ent_mean)

            # Total loss combines all terms
            loss = (
                p_loss
                + cfg["value_coef"] * v_loss
                + entropy_loss
                + cfg["aux_yaw_coef"] * yaw_loss
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()

            # Gradient clipping for training stability
            grad_norm = nn.utils.clip_grad_norm_(policy.parameters(), cfg["max_grad_norm"]).item()
            optimizer.step()

            # Accumulate metrics for logging
            metrics["policy_loss"] += p_loss.item()
            metrics["value_loss"] += v_loss.item()
            metrics["entropy"] += ent_mean.item()
            metrics["yaw_loss"] += yaw_loss.item()
            metrics["grad_norm"] += grad_norm
            n_updates += 1

    # Return mean metrics over all minibatch updates
    return {k: v / max(n_updates, 1) for k, v in metrics.items()}


### Step 5 — optuna objective (trial loop, logging, pruning, curriculum)

This section defines the `objective()` function that Optuna optimizes. For each trial, Optuna samples a set of PPO hyperparameters, trains the policy in the environment, and returns a final score.

main components inside the objective:

- hyperparameter sampling: Optuna selects values for learning rate, entropy coefficient, GAE lambda, epochs, batch size, and the yaw auxiliary loss weight.
- rollout + update loop: the agent collects rollouts of length `rollout_len` across `num_envs` parallel environments, then performs PPO updates with GAE advantages.
- curriculum progression: the stage increases when the rolling success rate (SSR) exceeds `advance_threshold` and enough steps have passed since the last advance. The success tolerance is updated per stage via `get_success_threshold()`.
- logging: prints progress to the console and stores metrics to TensorBoard and a CSV file per trial.
- pruning: trials are stopped early if they stagnate for too long at the same stage or fail to meet a minimum-stage schedule.
- checkpointing: periodic and final model snapshots are saved for later inspection.

note: the objective reports `max_stage_reached` to Optuna during training (`trial.report(...)`), but the returned scalar combines stage progress and current SSR (`final_score = max_stage_reached + ssr`).


In [ ]:
def objective(trial, env, base_log_dir, get_success_threshold):
    """Run one Optuna trial and return a scalar score (higher is better)."""

    # -------------------------------------------------------------------------
    # variable hyperparameters (sampled by Optuna for this trial)
    # -------------------------------------------------------------------------
    learning_rate = trial.suggest_float("learning_rate", 5e-5, 3e-4, log=True)
    entropy_coef  = trial.suggest_float("entropy_coef", 0.003, 0.015, log=True)
    gae_lambda    = trial.suggest_float("gae_lambda", 0.90, 0.98)
    epochs        = trial.suggest_int("epochs", 3, 7)
    batch_size    = trial.suggest_categorical("batch_size", [1024, 2048])
    aux_yaw_coef  = trial.suggest_float("aux_yaw_coef", 0.15, 0.45)

    # merge fixed params + sampled params into a single config dict
    cfg = {
        **FIXED_PARAMS,
        "learning_rate": learning_rate,
        "entropy_coef": entropy_coef,
        "gae_lambda": gae_lambda,
        "epochs": epochs,
        "batch_size": batch_size,
        "aux_yaw_coef": aux_yaw_coef,
    }

    # -------------------------------------------------------------------------
    # env + model init
    # -------------------------------------------------------------------------
    device = torch.device("cuda:0")
    env.set_curriculum_level(0)  # always start trials from stage 0

    policy = VisionIMUAttentionYawPolicy().to(device)
    optimizer = torch.optim.Adam(policy.parameters(), lr=cfg["learning_rate"])

    # -------------------------------------------------------------------------
    # logging setup (TensorBoard + per-trial CSV)
    # -------------------------------------------------------------------------
    writer = None
    csv_file = None
    csv_writer = None
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    if HAS_TENSORBOARD:
        trial_log_dir = f"{base_log_dir}/trial_{trial.number}"
        os.makedirs(trial_log_dir, exist_ok=True)
        writer = SummaryWriter(trial_log_dir)

    csv_path = f"/home/schux00/logs/optuna_v4_T{trial.number}_{timestamp}.csv"
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    csv_file = open(csv_path, "w", newline="")
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow([
        "step", "stage", "ssr", "reward", "entropy", "yaw_loss",
        "policy_loss", "value_loss", "success_threshold_cm", "hours"
    ])

    num_envs = cfg["num_envs"]
    rollout_len = cfg["rollout_len"]

    # -------------------------------------------------------------------------
    # rollout buffers (T, N, ...)
    # -------------------------------------------------------------------------
    rgb_buf        = torch.zeros((rollout_len, num_envs, 4, 128, 128), device=device)
    imu_buf        = torch.zeros((rollout_len, num_envs, 6), device=device)
    actions_buf    = torch.zeros((rollout_len, num_envs, 2), device=device)
    rewards_buf    = torch.zeros((rollout_len, num_envs), device=device)
    values_buf     = torch.zeros((rollout_len, num_envs), device=device)
    logprobs_buf   = torch.zeros((rollout_len, num_envs), device=device)
    dones_buf      = torch.zeros((rollout_len, num_envs), device=device)
    yaw_targets_buf = torch.zeros((rollout_len, num_envs, 1), device=device)
    priv_buf       = torch.zeros((rollout_len, num_envs, 7), device=device)

    # moving windows for reporting
    ep_rewards = deque(maxlen=300)
    stage_successes = deque(maxlen=300)
    cur_reward = torch.zeros(num_envs, device=device)

    # curriculum tracking
    current_stage = 0
    max_stage_reached = 0
    last_advance_step = 0
    last_stage_change_step = 0  # used for stagnation pruning

    # reset env and bookkeeping
    obs_dict, _ = env.reset()
    step = 0
    t0 = time.time()
    next_log = cfg["log_interval"]
    next_save = cfg["save_interval"]
    next_prune_check = OPTUNA_CONFIG["pruning_check_interval"]

    # privileged vector available? (asymmetric critic)
    has_privileged = ("privileged" in obs_dict) and (obs_dict["privileged"] is not None)

    # progressive success tolerance depends on stage
    success_threshold = get_success_threshold(current_stage)

    print("=" * 70)
    print(f"[TRIAL {trial.number}] TEKO Vision Optuna v4 - 50 Stages + Progressive Tolerance")
    print("=" * 70)
    print(f"Host: {socket.gethostname()}")
    print(f"LR: {cfg['learning_rate']:.6f} | Entropy: {cfg['entropy_coef']:.4f}")
    print(f"GAE Lambda: {cfg['gae_lambda']:.2f} | Batch: {cfg['batch_size']}")
    print(f"Epochs: {cfg['epochs']} | YawAux: {cfg['aux_yaw_coef']:.2f}")
    print(f"Max Stages: {cfg['max_stage']+1} (S0-S{cfg['max_stage']})")
    print(f"Initial Success Threshold: {success_threshold*100:.1f}cm")
    print("=" * 70)

    try:
        while step < OPTUNA_CONFIG["max_steps_per_trial"]:
            elapsed_h = (time.time() - t0) / 3600

            # hard walltime cap per trial
            if elapsed_h * 3600 > OPTUNA_CONFIG["max_walltime_s_per_trial"]:
                print(f"[T{trial.number}][TIME] Reached {OPTUNA_CONFIG['max_walltime_s_per_trial']//3600}h limit")
                break

            # ============================================================
            # rollout collection (T steps, N envs)
            # ============================================================
            for t in range(rollout_len):
                rgb = obs_dict["rgb"].to(device)      # (N, 4, 128, 128)
                imu = obs_dict["imu"].to(device)      # (N, 6)

                # privileged info (if provided by env)
                priv = obs_dict.get("privileged")
                if priv is not None:
                    priv = priv.to(device)
                    yaw_target = priv[:, 3:4]        # yaw target comes from privileged channel
                else:
                    yaw_target = torch.zeros(num_envs, 1, device=device)

                # act without gradient during rollout collection
                with torch.no_grad():
                    action, logp, value, _ = policy.act(rgb, imu, priv)

                # store rollout
                rgb_buf[t] = rgb
                imu_buf[t] = imu
                actions_buf[t] = action
                logprobs_buf[t] = logp
                values_buf[t] = value
                yaw_targets_buf[t] = yaw_target
                if priv is not None:
                    priv_buf[t] = priv

                # step env
                obs_dict, reward, term, trunc, info = env.step(action)
                done = term | trunc

                rewards_buf[t] = reward
                dones_buf[t] = done.float()
                cur_reward += reward

                # episode accounting for SSR / reward tracking
                if done.any():
                    done_idx = done.nonzero(as_tuple=False).squeeze(-1)

                    # success check: prefer env-provided success flag if present
                    if hasattr(env, "_last_success"):
                        succ = env._last_success.float()
                    else:
                        # fallback: compute distance-based success using current tolerance
                        _, _, sxy, _ = env.get_sphere_distances_from_physics()
                        succ = (sxy < success_threshold).float()

                    ep_rewards.extend(cur_reward[done_idx].cpu().tolist())
                    stage_successes.extend(succ[done_idx].cpu().tolist())
                    cur_reward[done_idx] = 0

                # global step counter counts env-steps (N per sim step)
                step += num_envs

            # ============================================================
            # compute GAE + PPO update
            # ============================================================
            with torch.no_grad():
                last_rgb = obs_dict["rgb"].to(device)
                last_imu = obs_dict["imu"].to(device)
                last_priv = obs_dict.get("privileged")
                if last_priv is not None:
                    last_priv = last_priv.to(device)
                _, _, last_value, _ = policy.act(last_rgb, last_imu, last_priv)

            advantages, returns = compute_gae(
                rewards_buf, values_buf, dones_buf,
                cfg["gamma"], cfg["gae_lambda"], last_value
            )

            metrics = ppo_update_with_yaw(
                policy, optimizer,
                rgb_buf, imu_buf, actions_buf, logprobs_buf,
                advantages, returns, yaw_targets_buf,
                priv_buf if has_privileged else None,
                cfg
            )

            ssr = float(np.mean(stage_successes)) if stage_successes else 0.0
            mean_r = float(np.mean(ep_rewards)) if ep_rewards else 0.0

            # ============================================================
            # curriculum progression
            # ============================================================
            if (
                len(stage_successes) >= 100 and
                ssr >= cfg["advance_threshold"] and
                step - last_advance_step >= cfg["min_steps_before_advance"] and
                current_stage < cfg["max_stage"]
            ):
                current_stage += 1
                max_stage_reached = max(max_stage_reached, current_stage)
                env.set_curriculum_level(current_stage)

                # update tolerance according to the progressive schedule
                success_threshold = get_success_threshold(current_stage)

                print(
                    f"[T{trial.number}][ADVANCE] Stage {current_stage-1} -> {current_stage} "
                    f"(SSR={ssr:.1%}, threshold={success_threshold*100:.1f}cm)"
                )

                stage_successes.clear()
                last_advance_step = step
                last_stage_change_step = step

                if writer:
                    writer.add_scalar("curriculum/stage", current_stage, step)
                    writer.add_scalar("curriculum/success_threshold_cm", success_threshold * 100, step)

            # ============================================================
            # logging + report to Optuna
            # ============================================================
            if step >= next_log:
                print(
                    f"[T{trial.number}][{step:,}] S{current_stage:02d} | SSR: {ssr:.1%} | R: {mean_r:.1f} | "
                    f"YawL: {metrics['yaw_loss']:.3f} | Ent: {metrics['entropy']:.3f} | "
                    f"MaxS: {max_stage_reached} | Thr: {success_threshold*100:.1f}cm | {elapsed_h:.1f}h"
                )

                if writer:
                    writer.add_scalar("train/ssr", ssr, step)
                    writer.add_scalar("train/reward", mean_r, step)
                    writer.add_scalar("train/entropy", metrics["entropy"], step)
                    writer.add_scalar("train/yaw_loss", metrics["yaw_loss"], step)
                    writer.add_scalar("train/policy_loss", metrics["policy_loss"], step)
                    writer.add_scalar("train/value_loss", metrics["value_loss"], step)
                    writer.add_scalar("train/grad_norm", metrics["grad_norm"], step)
                    writer.add_scalar("curriculum/stage", current_stage, step)
                    writer.add_scalar("curriculum/max_stage", max_stage_reached, step)
                    writer.add_scalar("curriculum/success_threshold_cm", success_threshold * 100, step)

                if csv_writer:
                    csv_writer.writerow([
                        step, current_stage, f"{ssr:.4f}", f"{mean_r:.2f}",
                        f"{metrics['entropy']:.4f}", f"{metrics['yaw_loss']:.4f}",
                        f"{metrics['policy_loss']:.4f}", f"{metrics['value_loss']:.4f}",
                        f"{success_threshold*100:.1f}", f"{elapsed_h:.2f}"
                    ])
                    csv_file.flush()

                # Optuna intermediate report (used for monitoring / pruning decisions)
                trial.report(max_stage_reached, step)

                next_log += cfg["log_interval"]

            # ============================================================
            # pruning (manual pruning even if Optuna pruner is disabled)
            # ============================================================
            if OPTUNA_CONFIG["pruning_enabled"] and step >= next_prune_check:
                # stagnation: no stage change for too long
                if step - last_stage_change_step > OPTUNA_CONFIG["stagnation_limit_steps"]:
                    print(
                        f"[T{trial.number}][STAGNATION] Stuck at S{current_stage} for "
                        f"{(step-last_stage_change_step)//1_000_000}M steps"
                    )
                    raise optuna.TrialPruned()

                # schedule-based pruning: must reach minimum stage by certain steps
                if should_prune(step, max_stage_reached):
                    print(f"[T{trial.number}][PRUNE] at step {step:,} with max_stage={max_stage_reached}")
                    raise optuna.TrialPruned()

                next_prune_check += OPTUNA_CONFIG["pruning_check_interval"]

            # ============================================================
            # checkpointing
            # ============================================================
            if step >= next_save:
                ckpt_path = f"/home/schux00/checkpoints/optuna_v4_T{trial.number}_S{current_stage}_{step//1000}k.pt"
                os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
                torch.save({
                    "trial": trial.number,
                    "step": step,
                    "stage": current_stage,
                    "max_stage": max_stage_reached,
                    "policy": policy.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "config": cfg,
                    "success_threshold": success_threshold,
                }, ckpt_path)
                print(f"[T{trial.number}][SAVE] {ckpt_path}")
                next_save += cfg["save_interval"]

            # ============================================================
            # early stop if the final stage is solved reliably
            # ============================================================
            if current_stage >= cfg["max_stage"] and ssr >= 0.95:
                print("=" * 70)
                print(f"[T{trial.number}][SUCCESS] Reached Stage {cfg['max_stage']} with SSR={ssr:.1%}!")
                print(f"[T{trial.number}][SUCCESS] Final threshold: {success_threshold*100:.1f}cm (1cm precision)")
                print("=" * 70)
                break

    except optuna.TrialPruned:
        # let Optuna handle pruned trials normally
        raise

    except Exception as e:
        # treat unexpected errors as pruned (keeps study running)
        print(f"[T{trial.number}][ERROR] {repr(e)}")
        import traceback
        traceback.print_exc()
        raise optuna.TrialPruned()

    finally:
        # reset env stage so other trials start clean
        env.set_curriculum_level(0)

        # always save a final snapshot for this trial (even if pruned)
        final_path = f"/home/schux00/checkpoints/optuna_v4_T{trial.number}_FINAL_S{max_stage_reached}.pt"
        torch.save({
            "trial": trial.number,
            "step": step,
            "stage": current_stage,
            "max_stage": max_stage_reached,
            "policy": policy.state_dict(),
            "config": cfg,
            "success_threshold": success_threshold,
        }, final_path)
        print(f"[T{trial.number}][FINAL] Saved to {final_path}")

        # store hyperparams in TensorBoard hparams tab
        if writer:
            writer.add_hparams(
                {
                    "lr": learning_rate,
                    "entropy": entropy_coef,
                    "gae_lambda": gae_lambda,
                    "epochs": epochs,
                    "batch_size": batch_size,
                    "aux_yaw": aux_yaw_coef,
                },
                {"hparam/max_stage": max_stage_reached}
            )
            writer.close()

        if csv_file:
            csv_file.close()

    # final score used by Optuna "direction=maximize"
    print(f"[T{trial.number}][DONE] MaxStage={max_stage_reached}, Steps={step:,}, Time={(time.time()-t0)/3600:.1f}h")
    final_score = float(max_stage_reached) + ssr
    print(f"[T{trial.number}][SCORE] Stage={max_stage_reached} + SSR={ssr:.1%} = {final_score:.2f}")
    return final_score


### Step 6 — worker execution (environment setup, study creation, trial loop)

This section defines the worker that runs Optuna trials on a machine (or cluster node).

what it does:

- reproducibility: sets a per-worker random seed (base seed + time offset) for PyTorch, NumPy, and Python.
- simulation boot: launches Isaac Lab using `AppLauncher` in headless mode with cameras enabled.
- environment setup: creates the tiled IMU vision environment with:
  - `num_envs` taken from the fixed PPO configuration
  - curriculum enabled
  - asymmetric critic enabled (privileged observations for the critic only)
- curriculum integration: monkey-patches the environment reset function so that each reset applies the unified curriculum logic (`reset_environment_curriculum`) and initializes bookkeeping buffers.
- optuna study: creates/loads a study backed by a SQLite database and uses an NSGA-II sampler for exploration of the search space.
- logging: creates a TensorBoard directory for trial logs and prints key runtime info (host, seed, storage path, curriculum schedule).
- optimization loop: repeatedly calls `study.optimize(...)` until reaching the target number of trials, then closes the environment and simulator cleanly.


In [ ]:
def run_worker(args):
    # speed: allow cuDNN to pick best kernels for fixed input sizes
    torch.backends.cudnn.benchmark = True

    # per-worker seed (base seed + small time-based offset to diversify workers)
    seed = args.seed + int(time.time()) % 1000
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # launch Isaac Lab / Isaac Sim
    app = AppLauncher(args)
    sim = app.app

    # ensure TEKO package is importable
    sys.path.insert(0, "/workspace/teko/source/teko")

    # -------------------------------------------------------------------------
    # environment imports + curriculum utilities
    # -------------------------------------------------------------------------
    from teko.tasks.direct.teko.teko_env_tiled_imu import TekoEnvTiledIMU
    from teko.tasks.direct.teko.teko_env_cfg import TekoEnvCfg

    from teko.tasks.direct.teko.curriculum.curriculum_teko import (
        reset_environment_curriculum,
        set_curriculum_level,
        get_success_threshold,
        MAX_STAGE,
    )

    # -------------------------------------------------------------------------
    # environment configuration
    # -------------------------------------------------------------------------
    cfg = TekoEnvCfg()
    cfg.scene.num_envs = FIXED_PARAMS["num_envs"]  # keep consistent with PPO config
    cfg.enable_curriculum = True
    cfg.asymmetric_critic = True                   # critic may use privileged state

    env = TekoEnvTiledIMU(cfg=cfg)

    # -------------------------------------------------------------------------
    # monkey-patch reset to enforce unified curriculum on every reset
    # (useful if env originally had a different curriculum/reset logic)
    # -------------------------------------------------------------------------
    original_reset_idx = env._reset_idx

    def patched_reset_idx(env_ids):
        # call the base class reset implementation (keeps physics/state consistent)
        type(env).__bases__[0]._reset_idx(env, env_ids)
        env._lazy_init_articulation()

        # init bookkeeping tensors if needed
        num_envs = env.scene.cfg.num_envs
        if env.prev_distance is None:
            env.prev_distance = torch.zeros(num_envs, device=env.device)
        if env.prev_actions is None:
            env.prev_actions = torch.zeros((num_envs, 2), device=env.device)
        if env.step_count is None:
            env.step_count = torch.zeros(num_envs, dtype=torch.int32, device=env.device)

        # reset per-episode trackers for selected envs
        env.prev_actions[env_ids] = 0.0
        env.step_count[env_ids] = 0

        if env.frame_counts is not None:
            env.frame_counts[env_ids] = 0

        # apply unified curriculum reset logic (positions, offsets, noise, etc.)
        reset_environment_curriculum(env, env_ids)

        # refresh "previous distance" baseline after reset
        _, _, surface_xy, _ = env.get_sphere_distances_from_physics()
        env.prev_distance[env_ids] = surface_xy[env_ids]

    env._reset_idx = patched_reset_idx

    # expose a curriculum setter compatible with the training loop
    env.set_curriculum_level = lambda level: set_curriculum_level(env, level)

    # -------------------------------------------------------------------------
    # Optuna study (SQLite storage + NSGA-II sampler)
    # -------------------------------------------------------------------------
    storage = OPTUNA_CONFIG["storage_path"]
    os.makedirs(os.path.dirname(storage.replace("sqlite:///", "")), exist_ok=True)

    study = optuna.create_study(
        study_name=OPTUNA_CONFIG["study_name"],
        storage=storage,
        direction="maximize",
        load_if_exists=True,
        sampler=optuna.samplers.NSGAIISampler(population_size=20, seed=seed),
        pruner=optuna.pruners.NopPruner(),  # pruning handled manually inside objective()
    )

    # -------------------------------------------------------------------------
    # TensorBoard base directory for this worker run
    # -------------------------------------------------------------------------
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_log_dir = f"/home/schux00/tensorboard/vision_optuna_v4_{timestamp}"
    os.makedirs(base_log_dir, exist_ok=True)

    print("=" * 70)
    print("TEKO Vision Optuna v4 - 50 Stages + Progressive Tolerance")
    print("=" * 70)
    print(f"Host: {socket.gethostname()} | Seed: {seed}")
    print(f"Storage: {storage}")
    print(f"Dashboard: optuna-dashboard {storage}")
    print(f"TensorBoard: {base_log_dir}")
    print(f"Max Stages: {MAX_STAGE + 1} (S0-S{MAX_STAGE})")
    print("Tolerance: 3cm (S0-20) -> 2cm (S21-30) -> 1.5cm (S31-41) -> 1cm (S42-49)")
    print(f"Trials so far: {len(study.trials)}")
    print("=" * 70)

    # -------------------------------------------------------------------------
    # run trials until reaching the target number of trials
    # -------------------------------------------------------------------------
    try:
        while len(study.trials) < OPTUNA_CONFIG["target_total_trials"]:
            study.optimize(
                lambda tr: objective(tr, env, base_log_dir, get_success_threshold),
                n_trials=1
            )
    finally:
        # always close resources cleanly (important on cluster runs)
        env.close()
        sim.close()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--seed", type=int, default=42)
    AppLauncher.add_app_launcher_args(parser)

    args = parser.parse_args()
    args.headless = True
    args.enable_cameras = True
    run_worker(args)


if __name__ == "__main__":
    main()
